# 使用 GPT-5.5 评估有据可依的空间推理

传统上，计算机视觉系统通过将空间理解分解为更窄的感知任务来处理它：检测物体、分割区域、对场景分类，或将语言定位到边界框和掩码上。当目标是定位时，这种分解效果很好。但当任务要求模型解读视觉场景、推理其结构含义、应用约束并生成一个始终有效的方案时，这种方式就不那么直接了。

前沿多模态模型让另一种工作流成为可能。我们不必针对某个布局领域，在大量成对样本上训练一个特定任务模型，而是可以测试：在给予足够的支撑时，像 GPT-5.5 这样具备推理能力的模型能否直接解决任务的更多部分。这些支撑包括：清晰的视觉上下文、一套操作流程、受约束的对象、结构化的输出格式，以及能够指出结果在哪里出问题的 evals。

本文的用例是根据空白平面图生成办公室布局。给定一张空白平面图图像、一份办公室布局 SOP 和一份家具目录，GPT-5.5 会生成一份结构化的布局规格（spec），其中识别空间、推断房间用途、应用数值规则、摆放家具，并记录对源图纸的假设。这个想法也来自一个真实的办公室问题：我们纽约办公室的人数增长得太快，以至于“能不能在哪里再塞几张桌子？”开始听起来像是一个可以合理地问模型的问题。

我们选择办公室平面图作为测试用例，是因为其失败情况很容易检查。GPT-5.5 接收一张空白平面图、一份可复用的布局 SOP 和一份小型家具目录，然后生成结构化的布局规格。该规格识别空间、推断房间用途、应用数值规则、摆放家具，并记录对源图纸的假设。

整个实验围绕评估这份规格展开，而不是仅仅评判渲染出的图像。每次运行都可以检查其机械有效性、语义依据，以及（在存在人工编写的参考时）与参考的相似度。这让我们能够区分那些在视觉效果图中会混为一谈的失败模式：无效的几何、错误的房间解读、错误的家具配置，以及薄弱的源图依据。

## 任务设置：从平面图到布局规格

生成步骤将问题拆分为三种输入：1）来自平面图的视觉证据，2）来自 SOP 的规划策略，3）来自家具目录的对象尺寸。

可见的证据是空白平面图图像 `empty.png`。模型从图像本身推断墙体、门、楼梯、洗手间、隔断、开口以及可用的房间形状。在生成期间，它看不到已填充的参考图像或黄金 JSON；这些产物会保留下来用于评估。

<p align="center">
  <img src="images/grounded_spatial_figure_1_empty_vs_reference.png" alt="Figure 1: empty floorplan and hand-filled reference" width="90%">
</p>
<p align="center"><em>图 1：生成期间模型只能看到空白平面图。已填充的参考图保留用于评估和定性比较。</em></p>

`constraints.json` 中可复用的 SOP 定义了规划策略。它描述了布局应包含的功能空间、这些空间所需的核心家具，以及方案应尽量满足的空间规则。

```json
{
  "profile_id": "office_furniture_sop_v1",
  "client_brief": {
    "required_spaces": [
      { "semantic_type": "open_office", "count": 2 },
      { "semantic_type": "reception", "count": 1 },
      { "semantic_type": "executive_suite", "count": 1 }
    ],
    "required_objects": [
      { "object_type": "reception_desk", "count": 1, "preferred_semantic_type": "reception" },
      { "object_type": "executive_desk", "count": 1, "preferred_semantic_type": "executive_suite" },
      { "object_type": "guest_chair", "count": 3, "preferred_semantic_type": "executive_suite" }
    ]
  },
  "spatial_constraints": [
    {
      "id": "primary_circulation_clearance",
      "enforcement": "hard",
      "primary_path_min_clearance_in": 48,
      "cluster_spacing_min_in": 42
    },
    {
      "id": "workstation_cluster_size",
      "enforcement": "soft",
      "allowed_cluster_sizes": [4, 6]
    },
    {
      "id": "storage_ratio",
      "enforcement": "soft",
      "rule": "Prefer roughly 1 storage cabinet for every 8 open-office workstations, rounded up."
    }
  ]
}
```
家具目录提供对象尺寸。它把 `workstation`、`reception_desk` 或 `storage_cabinet` 这样的标签转化为带有尺寸和简单摆放先验的对象，因此模型必须推理这些对象是否真的放得下。

```json
{
  "workstation": {
    "dimensions_in": [60, 30],
    "rules": ["keep grouped", "orient for circulation"]
  },
  "reception_desk": {
    "dimensions_in": [72, 30],
    "rules": ["place near entrance", "keep visitor path clear"]
  },
  "executive_desk": {
    "dimensions_in": [84, 36],
    "rules": ["prefer private enclosed room", "keep guest-facing approach clear"]
  },
  "guest_chair": {
    "dimensions_in": [28, 28],
    "rules": ["use for visitor seating", "keep balanced around executive or meeting furniture"]
  },
  "storage_cabinet": {
    "dimensions_in": [48, 20],
    "rules": ["prefer wall adjacency", "avoid circulation"]
  }
}
```
这种分离让设置可以在不同案例间复用。新的平面图可以改变几何形状，而 SOP 和目录保持不变，这使任务更多地考验空间推理，而不是针对具体案例的提示。

在当前实现中，生成过程分为两个推理阶段。首先，GPT-5.5 从图像中确定房间语义。然后，它基于该语义房间方案和固定的核心要求生成最终布局。这种拆分来自爬坡式迭代（hillclimb）：识别房间配置和摆放家具虽然相关，但并不是完全相同的推理问题。

主要输出是 `raw_layout.json`。它是一份可由机器检查的空间规格，可以在渲染之前进行验证、修复、编译和检查。该规格包括可行性判断、假设、未满足的约束、语义分区、家具摆放和通行路径。摆放信息包括对象类型、中心坐标、尺寸、正交旋转、所属分区、可选分组、理由和置信度。

所有坐标和尺寸都以英寸为单位，旋转是离散的：`0`、`90`、`180` 或 `270`。这让输出贴近家具规划的实践，也更便于用几何检查进行验证。

渲染处于下游。GPT Image 2 可以使用原始平面图以及从结构化规格派生出的指令，生成用于检查的可视化产物。渲染图让人更容易审阅方案，但权威来源是规格。评估始终以规格为先，因为有效性是几何层面的。

## 评估设置

生成和评估作为两个独立阶段处理。GPT-5.5 首先生成候选布局规格 `raw_layout.json`。然后 Promptfoo 对保存下来的规格执行评估。配置中使用了 echo provider，因为布局已经生成好了；Promptfoo 并不创建方案，只负责协调对候选 JSON 的各项检查。

每个候选都会从三个方面进行检查：

1. **validity：确定性的硬性关卡。**
这是唯一的通过/失败关卡。它检查生成的布局相对于 constraints.json 和源平面图是否在机械上有效。检查涵盖硬性空间约束，例如与源墙体的碰撞、对象重叠、摆放是否位于声明的分区内，以及其他无论渲染图看起来多么合理都会让方案无法使用的几何错误。

2. **grounded_program：语义依据得分。**
该得分结合了确定性的房间覆盖检查，以及 GPT-5.5 对布局是否忠实于可见平面图和需求说明的判断。它考察是否识别出了正确类型的房间、主要空间是否得到了恰当的家具配置，以及分区解读是否以图纸为依据而非凭空捏造。这是一个得分而非硬性否决规则，因为语义解读更难归结为单一的确定性检查。

3. **reference_similarity：确定性的黄金规格得分。**
当存在黄金 JSON 规格时，该检查会通过分区一致性、家具配置一致性和容差范围内的摆放接近度，将生成的布局与参考进行比较。这为我们提供了一个基准式的信号，而不会让人工参考成为唯一可接受的布局。它对于比较候选方案很有用，但不是硬性关卡。

每次运行的输出都会写入 `eval_results.json`，并在追踪查看器中展示。查看器有意保持简单，但它让评估循环变得具体：每次运行都会在一个地方展示案例、模型、推理设置、渲染状态、有效性结果、语义依据得分和参考相似度得分。

<p align="center">
  <img src="images/grounded_spatial_failed_run_viewer.png" alt="Example failed run in the trace viewer" width="90%">
</p>
<p align="center"><em>查看器中展示的一次失败运行示例。</em></p>

这个候选方案未能通过硬性的 `validity` 关卡，因为行政办公桌和访客椅与源墙体或隔断线条发生了重叠。它仍然分别获得了 `grounded_program` 和 `reference_similarity` 的软性得分，这让失败比单一的通过/失败标签更容易诊断。

这种分离是一项关键的设计选择，因为一次运行可能在机械上无效，但在语义上仍有参考价值。它可能满足硬性几何检查，却选错了房间配置。它也可能在家具数量上与黄金规格相似，但在摆放或分区上有所偏离。将这些信号分开，让我们能够对系统进行爬坡式迭代：根据失败发生在哪一层，改进提示词、修订规格、添加验证器、修复摆放问题，或收集更好的黄金参考。

## 结果：精选的迭代检查点

最有价值的运行并不是一串干净的渐进式改进，而是暴露了工作流中不同问题的检查点。其规律是：错误的指标、错误的分解、错误的约定，最后在模型推理和确定性几何之间实现了更好的分工。

### 第 1 次运行：有效的布局暴露了错误的指标

一个早期案例通过了硬性检查，并生成了一个连贯的办公室方案。GPT-5.5 选择东侧的大房间作为开放办公区，将中央大厅作为接待区，并将北侧一个较小的房间作为行政套间。在这种解读下，布局在内部是一致的。


<p align="center">
  <img src="images/grounded_spatial_run_1_valid_layout.png" alt="Run 1: empty floorplan, filled floorplan, and rendered floorplan" width="90%">
</p>
<p align="center"><em>生成的方案看起来合理，但与人工参考的视觉相似度是错误的主要指标。</em></p>

问题出在评估目标上。这次运行的参考得分只有 `0.0644`，因为我们当时仍在使用家具掩码 IoU 与人工图像进行比较。这使得得分对像素级重叠过于敏感，即使生成的方案是一个连贯的替代布局也是如此。

相比渲染图，JSON 让模型的解读更容易检查：

```json
{
  "assumptions": [
    "The broad double doors at the south edge of the central lobby are treated as the primary public entrance.",
    "The large east room is assigned as the usable open-office zone.",
    "Other unlabeled large rooms are left unprogrammed rather than classified as open-office because the source image provides no room labels."
  ],
  "zones": [
    {
      "id": "zone_open_office_east",
      "label": "East open-office room used for workstation density calculation"
    }
  ]
}
```
这推动项目转向以规格为先的评估。渲染图对检查依然有用，但评估的主要对象必须是结构化方案：模型选择了哪些分区、分配了什么配置，以及布局在机械上是否有效。

### 第 2 次运行：看似合理的布局暴露了错误的分解

在一张更复杂的办公室平面图上，生成的家具方案看起来合理，但房间解读是错的。接待区被放到了错误的区域，南侧的独立办公区没有保留下来，候选方案得出的房间分解与参考配置存在实质性差异。


<p align="center">
  <img src="images/grounded_spatial_run_2_wrong_decomposition.png" alt="Run 2: empty floorplan, filled floorplan, and rendered floorplan" width="90%">
</p>
<p align="center"><em>可见的家具方案看起来合理，但房间配置的解读在摆放之前就已经出现了偏差。</em></p>

这改变了工作流。我们将任务拆分为两个阶段，也就是两次 LLM 调用：首先从图像中确定房间语义，然后基于该语义房间方案和固定的核心要求生成家具布局。我们还添加了明确的客户需求说明，这样模型就不必仅凭几何形状推断完整的业务配置。

### 第 8 次运行：过度填充的布局暴露了错误的约定

拆分之后，工作流变得更容易检查，但也短暂地变差了，而这种变差很有启发意义。确定性的家具配置逻辑生成了一个包含 38 个工位的方案，因为需求在含义模糊的 `other` 房间之间传播得过于宽泛。

<p align="center">
  <img src="images/grounded_spatial_run_8_overfilled_layout.png" alt="Run 8: overfilled rendered floorplan" width="75%">
</p>
<p align="center"><em>过度规定的家具配置约定将工位需求传播得过于宽泛，导致出现了一个不切实际的 38 工位方案。</em></p>

这暴露了一个过度规定的问题。模型不再只是在解决空间任务；基准开始奖励对隐藏分配逻辑的复现。解决办法是将语义房间定位与家具配置约定分开，添加需求说明与空间的绑定，并减少对 `other` 这类宽泛标签的依赖。

目标变得更聚焦：让固定的核心要素保持确定性，同时为有效的替代布局保留足够的设计自由度。

### 第 15 次运行：一个有效的端到端案例厘清了分工

在后期表现最好的案例中，运行达到了物理有效性，尽管与参考房间配置之间仍存在一些语义分歧。候选方案在对一个分区边界问题进行一次模型修复后达到了 `completed_valid`，而确定性几何则通过多轮修复、正交旋转搜索、局部修复和一个小型联合重排兜底，清除了剩余的办公桌和储物柜碰撞。

<p align="center">
  <img src="images/grounded_spatial_run_15_trace_viewer.png" alt="Run 15: valid rendered floorplan and trace viewer" width="90%">
</p>
<p align="center"><em>在更难的办公楼案例上首次得到物理有效的结果。GPT-5.5 生成了有据可依的布局方案，确定性修复清除了最后的几何错误。</em></p>

这厘清了系统边界。GPT-5.5 在有据可依的语义决策上最有用：识别房间用途、分配配置，以及提出结构化方案。确定性代码则更适合低成本的几何清理：边界检查、对象重叠、源墙体碰撞、旋转和重排。

这就成了爬坡式迭代的方法。模型提出有据可依的布局规格。评估栈判断失败属于语义层面、机械层面还是参考层面。确定性代码修复那些不应再需要一次模型调用的部分。渲染图对检查依然有用，但进展是在规格层面衡量的。

## 要点总结

前沿模型显著改变了有据可依的空间规划工作流中切实可行的范围。它们可以读懂一张稀疏的平面图、推断房间配置、应用数值布局规则、输出结构化规格、暴露假设，并根据评估反馈做出调整，而无需针对这一具体布局问题进行特定任务的微调。

通过要求 GPT-5.5 输出可由机器检查的布局规格，系统将空间推理转化为一个可检查的产物：房间用途、假设、约束、摆放和修复点都会在渲染之前明确呈现。

这种表示方式也厘清了合理的分工。GPT-5.5 负责高层解读：读懂模糊的几何形状、分配房间配置，并提出连贯的布局。确定性组件则负责任务中应当精确的部分，包括边界检查、碰撞检测、净空规则和局部修复。因此，模型推理不会被视为自我验证的结果；它会通过周边系统受到约束、检查和改进。

剩下的挑战是语义依据。最难的平面图并不是那些没有有效解的，而是有好几个看似合理解的。在这些情况下，成功取决于系统能否识别出图纸所意图的配置，而不仅仅是能否生成物理上有效的布置。未来的进展将取决于更广泛的评估覆盖、更强的源几何提取能力，以及视觉分区与语义用途之间更精确的匹配。更宏观的结论是：GPT-5.5 能够让更多空间规划工作转向通用的多模态推理。